In [ ]:
%pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


def ask_llm(prompt, model="llama-3.3-70b-versatile"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,      # creativity level 0-1
        max_tokens=1024       # maximum response length
    )
    return response.choices[0].message.content

# Test it
response = ask_llm("Explain what a neural network is in 3 sentences")
print(response)


A neural network is a computer system inspired by the structure and function of the human brain, consisting of layers of interconnected nodes or "neurons" that process and transmit information. These nodes receive one or more inputs, perform a computation on those inputs, and then send the output to other nodes, allowing the network to learn and represent complex patterns in data. Through a process of training, where the network is fed large amounts of data and adjusted to minimize errors, a neural network can learn to perform tasks such as image recognition, speech recognition, and decision-making, often achieving high levels of accuracy and precision.


In [4]:
def ask_with_system(system_prompt, user_message,
                    model="llama-3.3-70b-versatile"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=1024
    )
    return response.choices[0].message.content

# Persona 1 — Expert Teacher
teacher_prompt = """You are an expert computer science teacher who explains
complex concepts simply. Always use analogies and real world examples.
Keep explanations under 150 words."""

# Persona 2 — Code Reviewer
reviewer_prompt = """You are a senior software engineer doing code review.
Be direct, specific, and brutally honest. Point out bugs, bad practices,
and suggest improvements. No sugarcoating."""

# Persona 3 — Interview Coach
coach_prompt = """You are a technical interview coach at a top tech company.
Evaluate answers critically. Point out what's missing, what's wrong,
and what would impress an interviewer."""

# Test all three personas with the same question
question = "What is gradient descent?"

print("=== TEACHER ===")
print(ask_with_system(teacher_prompt, question))
print("\n=== CODE REVIEWER ===")
code = """
def train(model, data):
    for x, y in data:
        pred = model(x)
        loss = (pred - y) ** 2
        loss.backward()
        optimizer.step()
"""
print(ask_with_system(reviewer_prompt, f"Review this code:\n{code}"))
print("\n=== INTERVIEW COACH ===")
answer = "Gradient descent is an optimization algorithm that minimizes loss"
print(ask_with_system(coach_prompt,
      f"I gave this answer in an interview: '{answer}'. How did I do?"))

=== TEACHER ===
Imagine you're at the top of a hill and want to reach the bottom. Gradient descent is like taking small steps down the hill, always moving in the direction of the steepest slope. With each step, you get closer to the bottom. In machine learning, the "hill" is a mathematical function, and the "steps" are adjustments to the model's parameters. The goal is to find the lowest point on the hill, which represents the best solution. By iteratively taking small steps in the direction of the steepest slope, gradient descent helps the model learn and improve its predictions.

=== CODE REVIEWER ===
**Code Review**

The provided code is a basic implementation of a training loop for a machine learning model. However, there are several issues and improvements that can be suggested:

### Issues:

1. **Undefined `optimizer`**: The `optimizer` variable is used but not defined anywhere in the code. This will result in a `NameError` when trying to call `optimizer.step()`.
2. **Missing `mo

In [5]:
class ConversationAssistant:
    def __init__(self, system_prompt, model="llama-3.3-70b-versatile"):
        self.model = model
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]

    def chat(self, user_message):
        # Add user message to history
        self.messages.append({
            "role": "user",
            "content": user_message
        })

        # Get response
        response = client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.7,
            max_tokens=1024
        )

        assistant_message = response.choices[0].message.content

        # Add assistant response to history
        self.messages.append({
            "role": "assistant",
            "content": assistant_message
        })

        return assistant_message

    def reset(self):
        self.messages = [self.messages[0]]  # keep system prompt
        print("Conversation reset.")

    def show_history(self):
        for msg in self.messages:
            role = msg['role'].upper()
            content = msg['content'][:100]
            print(f"[{role}]: {content}...")
        print()

# Create an AI tutor
tutor = ConversationAssistant("""
You are an AI engineering tutor. You teach concepts clearly,
ask follow up questions to check understanding, and give
practical examples. Remember everything the student tells you.
""")

# Multi turn conversation
print(tutor.chat("I want to learn about transformers"))
print()
print(tutor.chat("Can you give me a simple analogy?"))
print()
print(tutor.chat("How does this relate to what I just learned about attention?"))

Transformers are a fascinating topic in the field of artificial intelligence, particularly in natural language processing (NLP). They were introduced in a research paper by Vaswani et al. in 2017 and have since become a cornerstone of many state-of-the-art NLP models.

To start, can you tell me what you already know about transformers? Have you had any experience with NLP or deep learning before?

Also, what specific aspect of transformers are you interested in learning about? Are you looking to understand the underlying architecture, how they're used in specific applications like language translation or text classification, or something else? 

(By the way, I'll remember everything you tell me throughout our conversation, so feel free to share as much or as little as you'd like!)

Here's a simple analogy to help understand the basic concept of transformers:

Imagine you're trying to understand a sentence, and you're looking at each word individually. A traditional neural network might

In [6]:
def summarize(text, style="brief"):

    styles = {
        "brief": "Summarize in 2-3 sentences maximum.",
        "bullet": "Summarize as 5 bullet points.",
        "eli5": "Explain like I'm 5 years old.",
        "technical": "Give a detailed technical summary."
    }

    system = f"""You are an expert summarizer.
    {styles.get(style, styles['brief'])}
    Be concise and accurate."""

    return ask_with_system(system, f"Summarize this:\n\n{text}")

# Test with a real AI paper abstract
abstract = """
Attention mechanisms have become an integral part of compelling sequence
modeling and transduction models in various tasks, allowing modeling of
dependencies without regard to their distance in the input or output
sequences. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show
these models to be superior in quality while being more parallelizable
and requiring significantly less time to train.
"""

print("=== BRIEF ===")
print(summarize(abstract, "brief"))

print("\n=== BULLET POINTS ===")
print(summarize(abstract, "bullet"))

print("\n=== ELI5 ===")
print(summarize(abstract, "eli5"))

=== BRIEF ===
The Transformer is a new network architecture that relies solely on attention mechanisms, eliminating recurrence and convolutions. This model is shown to be superior in quality and more parallelizable than existing models. It also requires significantly less training time, as demonstrated in machine translation tasks.

=== BULLET POINTS ===
Here are 5 concise bullet points summarizing the text:

* Attention mechanisms are crucial in sequence modeling and transduction tasks.
* A new network architecture, the Transformer, is proposed, relying solely on attention mechanisms.
* The Transformer model eliminates recurrence and convolutions.
* Experiments show the Transformer model outperforms others in machine translation tasks.
* The Transformer model is more parallelizable and requires less training time.

=== ELI5 ===
Imagine you're talking to a robot. The robot needs to understand what you say, even if your sentence is very long. 

A new way to help the robot is called the 

In [7]:
class StudyAssistant:
    def __init__(self):
        self.client = client
        self.model = "llama-3.3-70b-versatile"
        self.conversation = []
        self.system = """You are an expert AI/ML study assistant helping
        a student prepare for AI engineering interviews. You explain concepts
        clearly, test understanding with questions, and give honest feedback.
        Always relate concepts to practical applications."""

    def explain(self, concept):
        prompt = f"Explain '{concept}' clearly with an example."
        return self._call(prompt)

    def quiz(self, topic):
        prompt = f"""Give me one Medium difficulty interview question about
        '{topic}'. After I answer, evaluate my response."""
        return self._call(prompt)

    def compare(self, thing1, thing2):
        prompt = f"Compare and contrast {thing1} vs {thing2}. When would you use each?"
        return self._call(prompt)

    def simplify(self, text):
        prompt = f"Explain this simply:\n{text}"
        return self._call(prompt)

    def _call(self, user_message):
        messages = [
            {"role": "system", "content": self.system},
            {"role": "user", "content": user_message}
        ]
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.7,
            max_tokens=1024
        )
        return response.choices[0].message.content

# Use it
assistant = StudyAssistant()

print("=== EXPLAIN ===")
print(assistant.explain("attention mechanism in transformers"))

print("\n=== COMPARE ===")
print(assistant.compare("Random Forest", "XGBoost"))

print("\n=== QUIZ ===")
print(assistant.quiz("neural networks"))

=== EXPLAIN ===
The attention mechanism in transformers is a crucial concept in Natural Language Processing (NLP) and deep learning.

**What is the attention mechanism?**

The attention mechanism is a technique used in transformers to help the model focus on specific parts of the input data when processing it. In traditional neural networks, the model processes the entire input sequence (e.g., a sentence) sequentially, using recurrent neural networks (RNNs) or convolutional neural networks (CNNs). However, this approach can be inefficient and lead to limitations, such as:

1.  **Sequential processing**: RNNs process the input sequence one step at a time, which can be slow for long sequences.
2.  **Fixed context**: RNNs use a fixed context window to capture dependencies, which can be insufficient for modeling complex relationships.

The attention mechanism addresses these limitations by allowing the model to:

1.  **Parallelize processing**: Process the entire input sequence simultaneou